# SpeakAI-Eval — Inference (Run Only)

Notebook này chỉ cần gọi lên và chạy. Đảm bảo bạn đã chạy **Setup Notebook** trước đó.
Mã nguồn, models và HF cache sẽ được đọc thẳng từ Kaggle Dataset.

In [ ]:
# Cài đặt thư viện
import subprocess, sys, os
try:
    import numpy
    np_ver = numpy.__version__
except: np_ver = '2.0.2'
print('Installing Rust (required for DeepFilterNet on Python 3.12)...')
os.system("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y")
os.environ['PATH'] = f"/root/.cargo/bin:{os.environ.get('PATH', '')}"
pkgs = [
    'torch>=2.1.0', 'torchaudio>=2.1.0', 'torch-geometric>=2.4.0',
    'transformers>=4.36.0', 'peft>=0.7.0', 'pyyaml>=6.0.1',
    'numpy>=1.24.0', 'soundfile>=0.12.1', 'nltk>=3.8.1',
    'python-dotenv>=1.0.0', 'deepfilternet', 'addict', 'modelscope',
    'speechbrain>=1.0.0', 'huggingface_hub>=0.23.0',
    'accelerate>=0.26.0', 'fastapi', 'uvicorn', 'python-multipart', 'nest-asyncio',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', '-q', f'numpy=={np_ver}'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=False)
print('Libraries installed!')
if not os.path.exists('/usr/local/bin/cloudflared'):
    print('Downloading cloudflared...')
    subprocess.run('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared', shell=True, check=True)
    subprocess.run('chmod +x /usr/local/bin/cloudflared', shell=True, check=True)


---
## Khởi tạo Pipeline từ Kaggle Input

In [ ]:
import sys, os
try:
    import peft.import_utils
    peft.import_utils.is_torchao_available = lambda: False
except Exception: pass
SOURCE_DIR = None
MODEL_DIR = None
PRONUNCIATION_PATH = None
L2_MDD_PATH = None
for search_dir in ['/kaggle/working', '/kaggle/input']:
    for root, dirs, files in os.walk(search_dir):
        if 'pretrained_models' in dirs and 'wavlm-large' in os.listdir(os.path.join(root, 'pretrained_models')):
            if MODEL_DIR is None: MODEL_DIR = root
        if 'infer' in dirs and 'speaker-diarize' in dirs:
            if SOURCE_DIR is None: SOURCE_DIR = root
        if 'pronunciation.pt' in files:
            if PRONUNCIATION_PATH is None: PRONUNCIATION_PATH = os.path.join(root, 'pronunciation.pt')
        if 'l2_mdd_best.pt' in files:
            if L2_MDD_PATH is None: L2_MDD_PATH = os.path.join(root, 'l2_mdd_best.pt')
if not MODEL_DIR:
    raise FileNotFoundError("Không tìm thấy thư mục 'pretrained_models/wavlm-large' trong dataset model.")
if not SOURCE_DIR:
    raise FileNotFoundError("Không tìm thấy source code (infer, speaker-diarize) trong dataset setup.")
if not PRONUNCIATION_PATH:
    raise FileNotFoundError("Không tìm thấy file 'pronunciation.pt' (mô hình speechocean) trong bất kỳ dataset nào.")
if L2_MDD_PATH:
    print(f'✅ Tìm thấy L2-MDD checkpoint: {L2_MDD_PATH}')
else:
    print('⚠️ Không tìm thấy l2_mdd_best.pt — chỉ dùng pronunciation model')
sys.path.insert(0, SOURCE_DIR)
sys.path.insert(0, f'{SOURCE_DIR}/speaker-diarize')
os.chdir(SOURCE_DIR)

# Lưu cache HuggingFace vào MODEL_DIR (nếu có thể) hoặc /kaggle/working
os.environ['HF_HOME'] = '/kaggle/working/hf_cache'

import builtins
import torch, yaml
builtins.torch = torch
import infer.pipeline as ip_mod
ip_mod.torch = torch
from infer.pipeline import SpeakingPipeline
from transformers import AutoModelForCausalLM, AutoTokenizer

print('Patching config dynamically...')
with open(f'{SOURCE_DIR}/configs/pronunciation.yaml', 'r') as f:
    cfg = yaml.safe_load(f)
cfg['asr']['model_name'] = f'{MODEL_DIR}/pretrained_models/whisper-large-v3'
cfg['wavlm']['model_name'] = f'{MODEL_DIR}/pretrained_models/wavlm-large'
cfg['paths']['pronunciation_checkpoint'] = PRONUNCIATION_PATH
if L2_MDD_PATH:
    cfg['paths']['l2_mdd_checkpoint'] = L2_MDD_PATH
os.makedirs('/kaggle/working/tmp_configs', exist_ok=True)
tmp_config = '/kaggle/working/tmp_configs/pronunciation.yaml'
with open(tmp_config, 'w') as f:
    yaml.dump(cfg, f)

print('Patching transcribe.py dynamically...')
import infer.transcribe
def patched_load_asr_config():
    with open(tmp_config, encoding='utf-8') as f:
        return yaml.safe_load(f).get('asr') or {}
infer.transcribe._load_asr_config = patched_load_asr_config

print('Patching speaker_diarize.embedding dynamically...')
import speaker_diarize.embedding as sd_emb
from modelscope.hub.snapshot_download import snapshot_download

def _resolve_eres_model():
    HUB_ID = 'damo/speech_eres2net_large_200k_sv_zh-cn_16k-common'
    for search_root in ['/kaggle/working', '/kaggle/input']:
        if os.path.exists(search_root):
            for root, dirs, files in os.walk(search_root):
                if 'pretrained_eres2net.pt' in files and 'configuration.json' in files:
                    p = os.path.join(root, 'pretrained_eres2net.pt')
                    if os.path.getsize(p) > 10_000_000:
                        print(f'✅ Found valid ERes2Net checkpoint: {root}')
                        return root
    print(f'⬇️ Downloading {HUB_ID} via ModelScope snapshot_download...')
    return snapshot_download(HUB_ID)

ERES_PATH = _resolve_eres_model()
sd_emb.DEFAULT_MODEL_ID = ERES_PATH
if hasattr(sd_emb, 'resolve_eres2net_model'):
    sd_emb.resolve_eres2net_model = lambda mid=None: ERES_PATH if (not mid or not os.path.isfile(os.path.join(mid, 'pretrained_eres2net.pt'))) else mid
_orig_eres_init = sd_emb.ERes2NetEmbedder.__init__
def _patched_eres_init(self, device='cuda', model_id=None):
    if not model_id or not os.path.isfile(os.path.join(model_id, 'pretrained_eres2net.pt')):
        model_id = ERES_PATH
    return _orig_eres_init(self, device=device, model_id=model_id)
sd_emb.ERes2NetEmbedder.__init__ = _patched_eres_init

print('Patching data.silence_split dynamically...')
import data.silence_split as ss_mod
_orig_ss_from_dict = ss_mod.SilenceSplitConfig.from_dict
def _safe_ss_from_dict(cfg):
    if isinstance(cfg, ss_mod.SilenceSplitConfig):
        return cfg
    if hasattr(cfg, '__dict__') and not isinstance(cfg, dict):
        cfg = vars(cfg)
    if not isinstance(cfg, dict):
        return ss_mod.SilenceSplitConfig()
    return _orig_ss_from_dict(cfg)
ss_mod.SilenceSplitConfig.from_dict = classmethod(lambda cls, cfg: _safe_ss_from_dict(cfg))

print('Patching infer.pipeline._build_dialogue dynamically...')
import infer.pipeline as ip_mod
def _patched_build_dialogue(teacher_sentences, student_sentences):
    turns = []
    for s in teacher_sentences:
        turns.append({
            'role': 'teacher',
            'scored': bool(s.get('scored') and s.get('scores')),
            'start_sec': s.get('start_sec'),
            'end_sec': s.get('end_sec'),
            'transcript': s.get('transcript', ''),
            'audio': s.get('audio'),
            'scores': s.get('scores'),
            'errors': s.get('errors'),
            'transformer_feedback': s.get('transformer_feedback'),
            'words_detail': s.get('words_detail'),
            'l2_mdd_feedback': s.get('l2_mdd_feedback'),
        })
    for s in student_sentences:
        turns.append({
            'role': 'student',
            'scored': True,
            'start_sec': s.get('start_sec'),
            'end_sec': s.get('end_sec'),
            'transcript': s.get('transcript', ''),
            'audio': s.get('audio'),
            'scores': s.get('scores'),
            'errors': s.get('errors'),
            'transformer_feedback': s.get('transformer_feedback'),
            'words_detail': s.get('words_detail'),
            'l2_mdd_feedback': s.get('l2_mdd_feedback'),
        })
    turns.sort(key=lambda t: (t.get('start_sec') or 0, 0 if t['role'] == 'teacher' else 1))
    student_turns = []
    last_teacher = None
    for t in turns:
        if t['role'] == 'teacher':
            last_teacher = t
        else:
            student_turns.append({
                **t,
                'teacher_prompt': last_teacher['transcript'] if last_teacher else None,
                'teacher_prompt_start_sec': last_teacher.get('start_sec') if last_teacher else None,
                'teacher_prompt_end_sec': last_teacher.get('end_sec') if last_teacher else None,
                'teacher_prompt_audio': last_teacher.get('audio') if last_teacher else None,
            })
    return {'turns': turns, 'student_turns': student_turns}
ip_mod._build_dialogue = _patched_build_dialogue

print('Patching WhisperTranscriber._generate for max_new_tokens safety...')
import infer.transcribe as it_mod
if hasattr(it_mod, 'WhisperTranscriber'):
    _orig_wt_gen = it_mod.WhisperTranscriber._generate
    def _patched_wt_gen(self, input_features):
        if not hasattr(self, 'max_new_tokens') or self.max_new_tokens > 440:
            self.max_new_tokens = 440
        return _orig_wt_gen(self, input_features)
    it_mod.WhisperTranscriber._generate = _patched_wt_gen

print('=== KHỞI TẠO PIPELINE PARALLELISM TRÊN 2 GPU (NVIDIA T4 x 2) ===')
print('  GPU 0 (cuda:0): Whisper-large-v3 ASR + TwoSpeakerSplitter + Registration Embedder')
print('  GPU 1 (cuda:1): SpeechOcean762 (WavLM-large + GAT) + L2-MDD (WavLM-large + CTC)')

print('Loading Registration Model on GPU 0 (cuda:0)...')
from speaker_diarize.embedding import ERes2NetEmbedder
extract_embedder = ERes2NetEmbedder(device='cuda:0')
print('Registration Embedder ready on cuda:0!')

print('Loading SpeakingPipeline (ASR & Diarize on cuda:0, Scoring on cuda:1)...')
pipeline = SpeakingPipeline(
    config_path=tmp_config,
    device='cuda:1',
    asr_device='cuda:0',
    diarize_device='cuda:0',
    l2_mdd_ckpt=L2_MDD_PATH,
)
print('Pipeline ready!')

print('Setup complete! 2 GPUs are actively sharing the workload.')


---
## Hàm Sinh Feedback Tổng Hợp

In [ ]:
def generate_turn_feedback(teacher_text, student_text, score, errors, l2_note=None):
    """Turn-level feedback based on SpeechOcean errors (standard IPA) and optional L2-MDD note."""
    bad_words = [w for w in errors.get('words', []) if w.get('score', 10) < 7.0]
    bad_ph = [p for p in errors.get('phonemes', []) if p.get('score', 10) < 7.0]
    parts = []
    if bad_words:
        word_strs = []
        for w in bad_words[:4]:
            ipa_str = f" ({w['word_ipa']})" if w.get('word_ipa') else ""
            word_strs.append(f"\"{w['word']}\"{ipa_str} ({w['score']:.1f})")
        parts.append(f"Từ phát âm yếu: {', '.join(word_strs)}")
    if bad_ph:
        ph_strs = []
        for p in bad_ph[:4]:
            ph_ipa = p.get('ipa') or f"/{p.get('phoneme', '')}/"
            w_text = p.get('word', '')
            w_ipa = p.get('word_ipa', '')
            in_word = f" trong \"{w_text}\" ({w_ipa})" if (w_text and w_ipa) else (f" trong \"{w_text}\"" if w_text else "")
            tip = p.get('tip', '')
            tip_str = f" — {tip}" if tip else ""
            ph_strs.append(f"{ph_ipa}{in_word} ({p['score']:.1f}){tip_str}")
        parts.append(f"Âm cần luyện: {'; '.join(ph_strs)}")
    res = ''
    if parts:
        res = '⚠️ Cần cải thiện: ' + '. '.join(parts)
    elif score >= 8.0:
        res = '✅ Phát âm tốt!'
    elif score >= 6.0:
        res = '👍 Phát âm khá, tiếp tục luyện tập.'
    if l2_note:
        res = f"{res}\n💡 {l2_note}" if res else f"💡 {l2_note}"
    return res

def generate_overall_summary(result):
    """Generate overall assessment summary using SpeechOcean scores."""
    otf = result.get('overall_transformer_feedback', {})
    if otf and otf.get('summary'):
        parts = [otf['summary']]
        if otf.get('tips'):
            parts.append('\n**💡 Gợi ý luyện tập:**')
            for tip in otf['tips'][:5]:
                parts.append(f'- {tip}')
        return '\n'.join(parts)
    
    student_sents = result.get('student', {}).get('sentences', [])
    if not student_sents:
        return 'Không có dữ liệu.'
    
    total_acc = sum(s.get('scores', {}).get('accuracy', 0) for s in student_sents) / len(student_sents)
    total_flu = sum(s.get('scores', {}).get('fluency', 0) for s in student_sents) / len(student_sents)
    total_pro = sum(s.get('scores', {}).get('prosodic', 0) for s in student_sents) / len(student_sents)
    overall_total = (total_acc + total_flu + total_pro) / 3
    
    # Determine level
    if overall_total >= 8.5: level = '🌟 Xuất sắc'
    elif overall_total >= 7.0: level = '👍 Tốt'
    elif overall_total >= 5.0: level = '⚡ Trung bình'
    elif overall_total >= 3.0: level = '⚠️ Yếu'
    else: level = '❌ Cần cải thiện nhiều'
    
    summary = f"""## {level} (SpeechOcean762)
\n**Tổng quan:** {overall_total:.1f}/10
- Chính xác (Accuracy): {total_acc:.1f}/10
- Trôi chảy (Fluency): {total_flu:.1f}/10
- Ngữ điệu (Prosody): {total_pro:.1f}/10
"""
    # Collect all tips from turns
    all_tips = set()
    for turn in result.get('dialogue', {}).get('turns', []):
        tf = turn.get('transformer_feedback', {})
        if tf and tf.get('tips'):
            for tip in tf['tips']:
                all_tips.add(tip)
    if all_tips:
        summary += '\n**💡 Gợi ý luyện tập:**\n'
        for tip in list(all_tips)[:6]:
            summary += f'- {tip}\n'
    
    return summary


---
## Khởi chạy Backend API (FastAPI + Cloudflare Tunnel)

In [ ]:
# Setup tunnel and inject globals into backend_api
import backend_api
import builtins

# Start Cloudflare Tunnel
PUBLIC_URL = backend_api.start_cloudflare_tunnel(8000)
backend_api.PUBLIC_URL = PUBLIC_URL
if PUBLIC_URL:
    backend_api._update_supabase_url(PUBLIC_URL)
    print(f'🚀 API IS LIVE AT: {PUBLIC_URL}')
else:
    print('Running without Cloudflare tunnel')

# Register notebook globals so backend_api can find them
builtins.pipeline = pipeline
builtins.extract_embedder = extract_embedder
builtins.generate_turn_feedback = generate_turn_feedback
builtins.generate_overall_summary = generate_overall_summary

# Start FastAPI server in background thread
import threading, uvicorn
config = uvicorn.Config(app=backend_api.app, host='0.0.0.0', port=8000, log_level='info')
server = uvicorn.Server(config)
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()
print('FastAPI Server is running!')
import time
try:
    while server_thread.is_alive():
        time.sleep(1)
except KeyboardInterrupt:
    print('Shutting down FastAPI Server...')
    server.should_exit = True
    server_thread.join(timeout=5)
